In [55]:
import pandas as pd
import numpy as np

# Read climate and financial data
df = pd.read_csv("../../02_processed_data/FINAL_CLIMATE_FINANCIAL_DATA.csv", index_col=0)
df = df.rename(columns={"index_nsa": "hpi"})
df.head()

/var/folders/26/4y4fzxhd7td390hxfyvrnw1m0000gp/T/ipykernel_16281/2703862107.py:5: DtypeWarning: Columns (16) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../../02_processed_data/FINAL_CLIMATE_FINANCIAL_DATA.csv", index_col=0)


,county_fips5,yr,hpi_change,hpi,state,flood_frequency,flood_property_damage,flood_duration_hours,heat_stress_index,drought_annual_mean_index,...,deaths_hurricane,injuries_hurricane,damage_hurricane,real_gdp,unemployment_rate,prop_rate,county_fips,total_home_purchase,total_amt_purchase,total_population
14,1001,2000,2.50,141.23,ALABAMA,1.0,200000.0,3.50,0.546615,-2.32,...,0.0,0.0,0.0,0,4.1,0.000,1001.0,831.0,72778000.0,44021.0
15,1001,2001,3.44,146.08,ALABAMA,2.0,37000.0,4.25,0.295779,1.43,...,0.0,0.0,0.0,1041239,4.1,0.000,1001.0,766.0,71592000.0,44889.0
16,1001,2002,2.67,149.98,ALABAMA,0.0,0.0,0.00,0.378973,-0.76,...,0.0,0.0,0.0,1080135,4.8,0.000,1001.0,730.0,78607000.0,45909.0
17,1001,2003,1.79,152.66,ALABAMA,2.0,58000.0,6.00,0.277611,2.39,...,0.0,0.0,0.0,1111465,5.0,0.245,1001.0,835.0,93128000.0,46800.0
18,1001,2004,2.92,157.12,ALABAMA,1.0,8000.0,3.50,0.411246,0.00,...,0.0,0.0,0.0,1283522,4.7,0.245,1001.0,908.0,111256000.0,48366.0


In [57]:
# Subset for data in California from 2018-2020
df_ca_2018_2020 = df[
    (df['yr'].between(2017, 2023)) & (df['state'] == 'CALIFORNIA') ]

In [59]:
df_ca_2018_2020.head()

,county_fips5,yr,hpi_change,hpi,state,flood_frequency,flood_property_damage,flood_duration_hours,heat_stress_index,drought_annual_mean_index,...,deaths_hurricane,injuries_hurricane,damage_hurricane,real_gdp,unemployment_rate,prop_rate,county_fips,total_home_purchase,total_amt_purchase,total_population
5427,6001,2017,7.04,1847.34,CALIFORNIA,25.0,0.0,80.92,0.377406,1.50,...,0.0,0.0,0.0,130430017,3.7,1.211,6001.0,13802.0,8.821017e+09,1659824.0
5428,6001,2018,8.97,2013.07,CALIFORNIA,17.0,0.0,8.13,0.267856,-2.01,...,0.0,0.0,0.0,135481671,3.1,1.230,6001.0,14179.0,9.439345e+09,1666596.0
5429,6001,2019,1.15,2036.27,CALIFORNIA,28.0,0.0,13.63,0.429874,0.28,...,0.0,0.0,0.0,139438051,3.0,1.241,6001.0,13728.0,9.617800e+09,1668412.0
5430,6001,2020,0.01,2036.42,CALIFORNIA,13.0,0.0,7.00,0.382561,-4.11,...,0.0,0.0,0.0,136360934,8.9,1.250,6001.0,13962.0,1.041741e+10,1680614.0
5459,6003,2017,12.09,213.93,CALIFORNIA,0.0,0.0,0.00,0.225889,3.68,...,0.0,0.0,0.0,114872,6.2,1.000,6003.0,9.0,2.816000e+06,1116.0


In [61]:
# Read data from California wildfire insurance
wildfire_insurance = pd.read_excel("../../02_processed_data/Residential-Property-Coverage-Amounts-Wildfire-Risk-and-Losses_Export.xlsx", 
sheet_name="Export_all", header=2)
wildfire_insurance.head()

,Source,Calendar Year,Policy Type,ZIP Code,Avg Cov A,Avg Cov C,Avg PPC Class,Avg Fire Risk,Count N,Count L,...,NonCat CovC Smoke Count,Cat CovA Fire Losses,Cat CovA Fire Count,Cat CovA Smoke Losses,Cat CovA Smoke Count,Cat CovC Fire Losses,Cat CovC Fire Count,Cat CovC Smoke Losses,Cat CovC Smoke Count,County
0,All Companies,18,CO,90001,54454.53,79376.32,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,Los Angeles
1,All Companies,19,CO,90001,123721.07,154296.95,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,Los Angeles
2,All Companies,20,CO,90001,23037.32,44099.54,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,Los Angeles
3,All Companies,18,DO,90001,230821.52,20971.27,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,Los Angeles
4,All Companies,19,DO,90001,243257.71,21405.82,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,Los Angeles


In [62]:
# Subset data from 2018-2020, with homeowner premium insurance type
wildfire_insurance['Calendar Year'] += 2000
wildfire_insurance_18_20 = wildfire_insurance[
    (wildfire_insurance['Policy Type'] == "HO") & (wildfire_insurance['Calendar Year'].between(2018, 2023))]

# Keep the column of interest
wildfire_insurance_18_20 = wildfire_insurance_18_20[["Calendar Year", "ZIP Code", "Earned Prem", "Earned Exp"]]

In [63]:
# Load crosswalk file
crosswalk = pd.read_excel("../../01_original_data/ZIP_COUNTY_122020.xlsx", dtype={'ZIP': str, 'COUNTY': str})
crosswalk.head()

,ZIP,COUNTY,RES_RATIO,BUS_RATIO,OTH_RATIO,TOT_RATIO
0,00501,36103,0.000000,1.000000,0.000000,1.000000
1,00601,72001,0.839799,0.800983,0.871166,0.838115
2,00601,72113,0.160201,0.199017,0.128834,0.161885
3,00602,72003,1.000000,0.998801,1.000000,0.999919
4,00602,72005,0.000000,0.001199,0.000000,0.000081


In [64]:
# Keep only the data from California
crosswalk_ca = crosswalk[crosswalk['ZIP'].str.startswith('9')]
crosswalk_ca.head()

,ZIP,COUNTY,RES_RATIO,BUS_RATIO,OTH_RATIO,TOT_RATIO
49760,90001,06037,1.0,1.0,1.0,1.0
49761,90002,06037,1.0,1.0,1.0,1.0
49762,90003,06037,1.0,1.0,1.0,1.0
49763,90004,06037,1.0,1.0,1.0,1.0
49764,90005,06037,1.0,1.0,1.0,1.0


In [65]:
# Force the zipcode to be a string
wildfire_insurance_18_20['ZIP Code'] = wildfire_insurance_18_20['ZIP Code'].astype(str)

# Keep only the largest res_ratio county for each zipcode to avoid 1 to many merge issue
crosswalk_ca['RES_RATIO'] = crosswalk_ca['RES_RATIO'].astype(float)
crosswalk_ca_sorted = crosswalk_ca.sort_values(by=['ZIP', 'RES_RATIO'], ascending=[True, False])
crosswalk_main = crosswalk_ca_sorted.drop_duplicates(subset=['ZIP'], keep='first')

# Merge the wildfire_insurance file with the crosswalk
wildfire_zip = wildfire_insurance_18_20.merge(
    crosswalk_main[['ZIP', 'COUNTY']], 
    left_on='ZIP Code', right_on='ZIP', 
    how='left'
)

/var/folders/26/4y4fzxhd7td390hxfyvrnw1m0000gp/T/ipykernel_16281/3822516419.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  crosswalk_ca['RES_RATIO'] = crosswalk_ca['RES_RATIO'].astype(float)


In [71]:
wildfire_zip.head()

,Calendar Year,ZIP Code,Earned Prem,Earned Exp,ZIP,COUNTY
0,2018,90001,1233018,1799,90001,06037
1,2019,90001,1351943,1863,90001,06037
2,2020,90001,1496492,1908,90001,06037
3,2021,90001,1629265,1976,90001,06037
4,2022,90001,1828773,2063,90001,06037


In [73]:
# Add the 0 to county fips code 
df_ca_2018_2020["county_fips5"] = df_ca_2018_2020["county_fips5"].astype(str).str.zfill(5)

wildfire_zip = wildfire_zip.rename(columns={"COUNTY": "county_fips5", "Calendar Year": "yr"})

# Merge and get the final dataframe
final_df = wildfire_zip.merge(
    df_ca_2018_2020,
    on=['county_fips5', 'yr'],
    how='left'
)

/var/folders/26/4y4fzxhd7td390hxfyvrnw1m0000gp/T/ipykernel_16281/3499580463.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ca_2018_2020["county_fips5"] = df_ca_2018_2020["county_fips5"].astype(str).str.zfill(5)


In [75]:
final_df = final_df.drop(columns=["ZIP"])

# Create premium per unit to show average earned premium for each housing unit
final_df = final_df[final_df['Earned Exp'] > 0]
final_df['prem_per_unit'] = final_df['Earned Prem'] / final_df['Earned Exp']

In [77]:
final_df.head()

,yr,ZIP Code,Earned Prem,Earned Exp,county_fips5,hpi_change,hpi,state,flood_frequency,flood_property_damage,...,injuries_hurricane,damage_hurricane,real_gdp,unemployment_rate,prop_rate,county_fips,total_home_purchase,total_amt_purchase,total_population,prem_per_unit
0,2018,90001,1233018,1799,06037,6.71,1566.79,CALIFORNIA,0.0,0.0,...,0.0,0.0,740088358,4.6,1.175,6037.0,52076.0,3.046519e+10,10061533.0,685.390773
1,2019,90001,1351943,1863,06037,3.50,1621.68,CALIFORNIA,0.0,0.0,...,0.0,0.0,767859358,4.5,1.182,6037.0,53991.0,3.281898e+10,10011602.0,725.680623
2,2020,90001,1496492,1908,06037,2.72,1665.85,CALIFORNIA,2.0,0.0,...,0.0,0.0,729068982,12.3,1.171,6037.0,55477.0,3.715788e+10,9996634.0,784.324948
3,2021,90001,1629265,1976,06037,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,824.526822
4,2022,90001,1828773,2063,06037,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,886.462918


In [117]:
import pandas as pd
import numpy as np

# =========================
# 0) Read climate/financial (COUNTY-YEAR)
# =========================
df = pd.read_csv(
    "../../02_processed_data/FINAL_CLIMATE_FINANCIAL_DATA.csv",
    index_col=0,
    low_memory=False
).copy()

df = df.rename(columns={"index_nsa": "hpi"})

# Keep CA + desired years
df_ca = df[(df["state"] == "CALIFORNIA") & (df["yr"].between(2017, 2023))].copy()

# Normalize keys
df_ca["yr"] = pd.to_numeric(df_ca["yr"], errors="coerce").astype("Int64")
df_ca["county_fips5"] = pd.to_numeric(df_ca["county_fips5"], errors="coerce").astype("Int64")
df_ca["county_fips5"] = df_ca["county_fips5"].astype(str).str.zfill(5)

# Enforce unique county-year (avoid accidental duplicates)
climate_cty = (
    df_ca
    .groupby(["county_fips5", "yr"], as_index=False)
    .mean(numeric_only=True)
)

# =========================
# 0.5) FIX: avoid using county TOTALS as if they are ZIP totals
# =========================
total_cols = ["total_population", "real_gdp", "total_home_purchase", "total_amt_purchase"]
have_totals = [c for c in total_cols if c in climate_cty.columns]

if "total_population" in climate_cty.columns:
    pop = climate_cty["total_population"]

    if "real_gdp" in climate_cty.columns:
        climate_cty["gdp_per_cap"] = np.where(pop.gt(0), climate_cty["real_gdp"] / pop, np.nan)

    if "total_home_purchase" in climate_cty.columns:
        climate_cty["home_purchase_per_cap"] = np.where(pop.gt(0), climate_cty["total_home_purchase"] / pop, np.nan)

    if "total_amt_purchase" in climate_cty.columns:
        climate_cty["amt_purchase_per_cap"] = np.where(pop.gt(0), climate_cty["total_amt_purchase"] / pop, np.nan)

if len(have_totals) > 0:
    climate_cty = climate_cty.drop(columns=have_totals)

print("Climate years:", sorted(climate_cty["yr"].dropna().unique()))

# =========================
# 1) Read insurance (ZIP-YEAR)
# =========================
wildfire_insurance = pd.read_excel(
    "../../02_processed_data/Residential-Property-Coverage-Amounts-Wildfire-Risk-and-Losses_Export.xlsx",
    sheet_name="Export_all",
    header=2
).copy()

wildfire_insurance["Calendar Year"] = pd.to_numeric(wildfire_insurance["Calendar Year"], errors="coerce")
wildfire_insurance["Calendar Year"] = np.where(
    wildfire_insurance["Calendar Year"] < 100,
    wildfire_insurance["Calendar Year"] + 2000,
    wildfire_insurance["Calendar Year"]
)

ins = wildfire_insurance[
    (wildfire_insurance["Policy Type"] == "HO") &
    (wildfire_insurance["Calendar Year"].between(2018, 2023))
].copy()

ins = ins[["Calendar Year", "ZIP Code", "Earned Prem", "Earned Exp"]].copy()
ins = ins.rename(columns={"Calendar Year": "yr", "ZIP Code": "zip"})

ins["zip"] = (
    ins["zip"].astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.extract(r"(\d{5})", expand=False)
    .str.zfill(5)
)

ins["yr"] = pd.to_numeric(ins["yr"], errors="coerce").astype("Int64")
ins["Earned Prem"] = pd.to_numeric(ins["Earned Prem"], errors="coerce")
ins["Earned Exp"]  = pd.to_numeric(ins["Earned Exp"], errors="coerce")

ins_zip = (
    ins
    .groupby(["zip", "yr"], as_index=False)
    .agg({"Earned Prem": "sum", "Earned Exp": "sum"})
)

ins_zip["prem_per_unit"] = np.where(
    ins_zip["Earned Exp"].gt(0),
    ins_zip["Earned Prem"] / ins_zip["Earned Exp"],
    np.nan
)

print("Ins years:", sorted(ins_zip["yr"].dropna().unique()))
print("Ins rows in 2021:", (ins_zip["yr"] == 2021).sum())

# =========================
# 2) Crosswalk ZIP -> COUNTY (main county per ZIP)
# =========================
crosswalk = pd.read_excel(
    "../../01_original_data/ZIP_COUNTY_122020.xlsx",
    dtype={"ZIP": str, "COUNTY": str}
).copy()

crosswalk_ca = crosswalk[crosswalk["ZIP"].str.startswith("9")].copy()
crosswalk_ca["RES_RATIO"] = pd.to_numeric(crosswalk_ca["RES_RATIO"], errors="coerce")

crosswalk_main = (
    crosswalk_ca
    .sort_values(["ZIP", "RES_RATIO"], ascending=[True, False])
    .drop_duplicates(subset=["ZIP"], keep="first")
    .rename(columns={"ZIP": "zip", "COUNTY": "county_fips5"})
)

crosswalk_main["zip"] = crosswalk_main["zip"].astype(str).str.zfill(5)
crosswalk_main["county_fips5"] = (
    pd.to_numeric(crosswalk_main["county_fips5"], errors="coerce")
    .astype("Int64")
    .astype(str)
    .str.zfill(5)
)

ins_zip = ins_zip.merge(crosswalk_main[["zip", "county_fips5"]], on="zip", how="left")

# =========================
# 3) Attach COUNTY climate to ZIP-YEAR insurance
# =========================
ins_zip_plus_climate = ins_zip.merge(
    climate_cty,
    on=["county_fips5", "yr"],
    how="left"
)

# =========================
# 4) Add 2017 climate-only ZIP rows
# =========================
zip_skeleton_2017 = crosswalk_main[["zip", "county_fips5"]].copy()
zip_skeleton_2017["yr"] = 2017

zip_skeleton_2017 = zip_skeleton_2017.merge(
    climate_cty,
    on=["county_fips5", "yr"],
    how="left"
)

for col in ["Earned Prem", "Earned Exp", "prem_per_unit"]:
    zip_skeleton_2017[col] = np.nan

# =========================
# 5) Final ZIP-YEAR dataset
# =========================
final_df = pd.concat([zip_skeleton_2017, ins_zip_plus_climate], ignore_index=True)
final_df = final_df[final_df["yr"].between(2017, 2023)].copy()

print("Final years:", sorted(final_df["yr"].dropna().unique()))
print("Final rows 2021-2023:", final_df[final_df["yr"].between(2021, 2023)].shape[0])

# ============================================================
# 6) NEW: create LAG-1 features so 2017 X predicts 2018 target
#    - For each zip: X_{t-1} -> predictors_lag1
#    - Target remains prem_per_unit in year t
# ============================================================

# Define target
TARGET = "prem_per_unit"

# Columns NOT to lag
id_cols = ["zip", "county_fips5", "yr", "Earned Prem", "Earned Exp", "prem_per_unit"]

# Pick candidate feature columns to lag (everything numeric except IDs/insurance amounts)
lag_feature_cols = [c for c in final_df.columns if c not in id_cols]

# Sort then create lag1 within zip
final_df = final_df.sort_values(["zip", "yr"]).copy()

for c in lag_feature_cols:
    final_df[f"{c}_lag1"] = final_df.groupby("zip")[c].shift(1)

# Training set for predicting 2018 uses lag1 from 2017:
# - Keep rows where target year is 2018+ (because needs lag1)
# - Keep only rows where lag features exist (lag1 not all missing)
model_df = final_df[final_df["yr"].between(2018, 2023)].copy()

# Optional: drop rows missing target (e.g., 2024 not present; also if some zips missing exp)
model_df = model_df.dropna(subset=[TARGET]).copy()

# Optional: drop rows where ALL lagged features are missing
lag_cols = [f"{c}_lag1" for c in lag_feature_cols]
model_df = model_df.dropna(subset=lag_cols, how="all").copy()

print("Model years:", sorted(model_df["yr"].unique()))
print("Model rows for 2018:", (model_df["yr"] == 2018).sum())

# Now:
# - model_df rows with yr==2018 have predictors from 2017 in *_lag1 columns
# - target is prem_per_unit for 2018
#
# Use X = model_df[lag_cols] (plus any non-lag features you want)
# y = model_df[TARGET]
#
# Example:
# X = model_df[lag_cols]
# y = model_df[TARGET]


Device: cpu
df original: (11668, 23) | unique zips: 4099
df_clean   : (9259, 23) | unique zips: 1864
Years in df_clean: [2017, 2018, 2019, 2020, 2021]
Base numeric covariates: 16
After lagging for X_{t-1} and dropping missing y: (7279, 39)
Lagged y-years available: [2018, 2019, 2020, 2021]
Train: (5469, 39) Val: (1827, 39) Test21: (1810, 39)
Num features used: 16
Using bounded output with y_scale = 5052.794746093741
Epoch 001 | train loss 546.421197 | val loss 510.488743


KeyboardInterrupt: 

In [113]:
final_df.columns

Index(['zip', 'county_fips5', 'yr', 'hpi_change', 'hpi', 'flood_frequency',
       'flood_property_damage', 'flood_duration_hours', 'heat_stress_index',
       'drought_annual_mean_index', 'fire_frequency', 'fire_size',
       'deaths_hurricane', 'injuries_hurricane', 'damage_hurricane',
       'unemployment_rate', 'prop_rate', 'county_fips',
       'home_purchase_per_cap', 'amt_purchase_per_cap', 'Earned Prem',
       'Earned Exp', 'prem_per_unit'],
      dtype='object')

In [115]:
from pathlib import Path

# go up 2 levels from current working directory
out_dir = Path.cwd().parents[1] / "02_processed_data"
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / "final_df_insurance_merged_data_in_california_2021_included.csv"
final_df.to_csv(out_path, index=False)

print("Saved to:", out_path.resolve())


Saved to: /Users/Jeep/Documents/duke_mss/stat_698/Capstone_Citizensbank_climate-risk_insurance-exposure_monitoring/02_processed_data/final_df_insurance_merged_data_in_california_2021_included.csv
